<a href="https://colab.research.google.com/github/Thays-Faleiro/api-risco-credito/blob/feature%2Festrutura-api-fastapi/Projeto_avaliativos_M%C3%B3dulo_2_Semana_13_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Célula 1: instalação

In [77]:
!pip install fastapi uvicorn pydantic scikit-learn pandas joblib httpx -q

Célula 2: pastas

In [78]:
import os
for p in ["data", "models", "app/schemas", "app/services", "app/routers"]:
    os.makedirs(p, exist_ok=True)
for p in ["app", "app/schemas", "app/services", "app/routers"]:
    open(f"{p}/__init__.py", "a").close()
print("Pastas criadas")

Pastas criadas


Célula 3: download dos dados

In [79]:
import os
from sklearn.datasets import fetch_openml

# Garante que o diretório de destino existe
os.makedirs("data", exist_ok=True)

df = fetch_openml("credit-g", version=1, as_frame=True).frame
df["inadimplente"] = (df["class"] == "bad").astype(int)
df = df.drop(columns=["class"])
df.to_csv("data/german_credit.csv", index=False)

print(df.shape)
for c in ["purpose", "housing", "checking_status"]:
    print(c, df[c].unique().tolist())

(1000, 21)
purpose ['radio/tv', 'education', 'furniture/equipment', 'new car', 'used car', 'business', 'domestic appliance', 'repairs', 'other', 'retraining']
housing ['own', 'for free', 'rent']
checking_status ['<0', '0<=X<200', 'no checking', '>=200']


Célula 4: treino (métricas)

In [80]:
import json, joblib, pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

df = pd.read_csv("data/german_credit.csv")
num = ["duration", "credit_amount", "age"]
cat = ["purpose", "housing", "checking_status"]
X, y = df[num + cat], df["inadimplente"]

X_tmp, X_te, y_tmp, y_te = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)
X_tr, X_va, y_tr, y_va = train_test_split(X_tmp, y_tmp, test_size=0.25, random_state=42, stratify=y_tmp)

pre = ColumnTransformer([
    ("num", Pipeline([("imp", SimpleImputer(strategy="median")), ("esc", StandardScaler())]), num),
    ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")),
                      ("ohe", OneHotEncoder(handle_unknown="ignore"))]), cat),
])
modelo = Pipeline([("pre", pre), ("clf", RandomForestClassifier(
    n_estimators=200, max_depth=8, class_weight="balanced", random_state=42))])
modelo.fit(X_tr, y_tr)

def avaliar(X_, y_):
    return {"acuracia": round(accuracy_score(y_, modelo.predict(X_)), 4),
            "f1_score": round(f1_score(y_, modelo.predict(X_)), 4),
            "roc_auc": round(roc_auc_score(y_, modelo.predict_proba(X_)[:, 1]), 4)}

metricas = {"validacao": avaliar(X_va, y_va), "teste": avaliar(X_te, y_te)}
print(json.dumps(metricas, indent=2))

joblib.dump(modelo, "models/modelo_pipeline.joblib")
json.dump(metricas, open("models/metricas.json", "w"), indent=2)

{
  "validacao": {
    "acuracia": 0.645,
    "f1_score": 0.4741,
    "roc_auc": 0.6827
  },
  "teste": {
    "acuracia": 0.79,
    "f1_score": 0.6719,
    "roc_auc": 0.8017
  }
}


Célula 5: schemas

In [81]:
%%writefile app/schemas/predicao.py
from typing import Literal
from pydantic import BaseModel, Field

class RequisicaoCredito(BaseModel):
    duration: float = Field(..., ge=1, le=120, description="Duração em meses", examples=[24])
    credit_amount: float = Field(..., gt=0, description="Valor do crédito", examples=[3500])
    age: float = Field(..., ge=18, le=120, description="Idade", examples=[35])
    purpose: Literal[
        "radio/tv", "education", "furniture/equipment", "new car", "used car",
        "business", "domestic appliance", "repairs", "other", "retraining",
    ] = Field(..., description="Finalidade do crédito", examples=["radio/tv"])
    housing: Literal["own", "for free", "rent"] = Field(
        ..., description="Tipo de moradia", examples=["own"])
    checking_status: Literal["<0", "0<=X<200", "no checking", ">=200"] = Field(
        ..., description="Situação da conta corrente", examples=["<0"])

class RespostaCredito(BaseModel):
    predicao: int = Field(..., description="0 = Baixo Risco, 1 = Alto Risco")
    probabilidade: float = Field(..., description="Probabilidade da classe predita")
    status_analise: str = Field(..., description="Resumo do resultado")

Overwriting app/schemas/predicao.py


Célula 6: serviço

In [82]:
%%writefile app/services/modelo_service.py
import pandas as pd
from app.schemas.predicao import RequisicaoCredito, RespostaCredito

class ModeloService:
    def __init__(self):
        self.modelo = None

    def definir_modelo(self, modelo):
        self.modelo = modelo

    def esta_carregado(self) -> bool:
        return self.modelo is not None

    def prever(self, dados: RequisicaoCredito) -> RespostaCredito:
        if not self.esta_carregado():
            raise RuntimeError("Modelo não está na memória.")
        df = pd.DataFrame([dados.model_dump()])
        try:
            classe = int(self.modelo.predict(df)[0])
            prob = float(self.modelo.predict_proba(df)[0][classe])
        except ValueError as erro:
            raise ValueError(f"Dados incompatíveis com o modelo: {erro}")
        msg = "Baixo Risco" if classe == 0 else "Alto Risco: análise manual necessária"
        return RespostaCredito(predicao=classe, probabilidade=round(prob, 4), status_analise=msg)

modelo_service = ModeloService()

Overwriting app/services/modelo_service.py


Célula 7: rota /health

In [83]:
%%writefile app/routers/health.py
from fastapi import APIRouter
from app.services.modelo_service import modelo_service

router = APIRouter(tags=["Monitoramento"])

@router.get("/health", status_code=200)
def health():
    return {"status": "operacional", "modelo_carregado": modelo_service.esta_carregado()}

Overwriting app/routers/health.py


Célula 8: rota /v1/predict

In [84]:
%%writefile app/routers/predict.py
from fastapi import APIRouter, HTTPException
from app.schemas.predicao import RequisicaoCredito, RespostaCredito
from app.services.modelo_service import modelo_service

router = APIRouter(prefix="/v1", tags=["Inferência"])

@router.post("/predict", response_model=RespostaCredito, status_code=200)
def predict(payload: RequisicaoCredito):
    try:
        return modelo_service.prever(payload)
    except ValueError as erro:
        raise HTTPException(status_code=400, detail=str(erro))
    except Exception:
        raise HTTPException(status_code=500, detail="Erro interno durante a inferência.")

Overwriting app/routers/predict.py


Célula 9: main.py

In [85]:
%%writefile app/main.py
import os, joblib
from contextlib import asynccontextmanager
from fastapi import FastAPI
from app.services.modelo_service import modelo_service
from app.routers import health, predict

CAMINHO_MODELO = os.getenv("CAMINHO_MODELO", "models/modelo_pipeline.joblib")

@asynccontextmanager
async def lifespan(app: FastAPI):
    if not os.path.exists(CAMINHO_MODELO):
        raise FileNotFoundError(f"Modelo não encontrado: {CAMINHO_MODELO}")
    modelo_service.definir_modelo(joblib.load(CAMINHO_MODELO))
    print("Modelo carregado na RAM")
    yield

app = FastAPI(title="API de Risco de Crédito", version="1.0.0", lifespan=lifespan)
app.include_router(health.router)
app.include_router(predict.router)

Overwriting app/main.py


Célula 10: testar a API (sem ngrok)

In [86]:
from fastapi.testclient import TestClient
from app.main import app

with TestClient(app) as client:
    print("HEALTH:", client.get("/health").status_code, client.get("/health").json())

    ok = {"duration": 24, "credit_amount": 3500, "age": 35,
          "purpose": "radio/tv", "housing": "own", "checking_status": "<0"}
    r = client.post("/v1/predict", json=ok)
    print("PREDICT:", r.status_code, r.json())

    ruim = {**ok, "age": 10}
    r = client.post("/v1/predict", json=ruim)
    print("INVALIDO:", r.status_code)

    invalido2 = {**ok, "housing": "castelo"}
    r = client.post("/v1/predict", json=invalido2)
    print("CATEGORIA INVALIDA:", r.status_code)

Modelo carregado na RAM
HEALTH: 200 {'status': 'operacional', 'modelo_carregado': True}
PREDICT: 200 {'predicao': 1, 'probabilidade': 0.758, 'status_analise': 'Alto Risco: análise manual necessária'}
INVALIDO: 422
CATEGORIA INVALIDA: 422


In [87]:
!cat app/schemas/predicao.py

from typing import Literal
from pydantic import BaseModel, Field

class RequisicaoCredito(BaseModel):
    duration: float = Field(..., ge=1, le=120, description="Duração em meses", examples=[24])
    credit_amount: float = Field(..., gt=0, description="Valor do crédito", examples=[3500])
    age: float = Field(..., ge=18, le=120, description="Idade", examples=[35])
    purpose: Literal[
        "radio/tv", "education", "furniture/equipment", "new car", "used car",
        "business", "domestic appliance", "repairs", "other", "retraining",
    ] = Field(..., description="Finalidade do crédito", examples=["radio/tv"])
    housing: Literal["own", "for free", "rent"] = Field(
        ..., description="Tipo de moradia", examples=["own"])
    checking_status: Literal["<0", "0<=X<200", "no checking", ">=200"] = Field(
        ..., description="Situação da conta corrente", examples=["<0"])

class RespostaCredito(BaseModel):
    predicao: int = Field(..., description="0 = Baixo Risco, 1 = Alto 

In [88]:
import importlib, sys

# remove módulos antigos da memória
for nome in [m for m in sys.modules if m.startswith("app")]:
    del sys.modules[nome]

from app.schemas.predicao import RequisicaoCredito
try:
    RequisicaoCredito(duration=24, credit_amount=3500, age=35,
                      purpose="radio/tv", housing="castelo", checking_status="<0")
    print("ACEITOU 'castelo' -> schema antigo em uso")
except Exception as e:
    print("REJEITOU 'castelo' -> schema novo OK")

REJEITOU 'castelo' -> schema novo OK


In [89]:
import app.schemas.predicao as p
print(p.__file__)
!ls -la app/schemas/

/content/app/schemas/predicao.py
total 16
drwxr-xr-x 3 root root 4096 Oct 23 23:27 .
drwxr-xr-x 6 root root 4096 Oct 23 23:27 ..
-rw-r--r-- 1 root root    0 Oct 23 23:30 __init__.py
-rw-r--r-- 1 root root 1171 Oct 23 23:31 predicao.py
drwxr-xr-x 2 root root 4096 Oct 23 23:27 __pycache__


In [90]:
!pip freeze | grep -iE "fastapi|uvicorn|pydantic|scikit-learn|pandas|numpy|joblib|httpx"
!python --version

fastapi==0.141.1
geopandas==1.2.0
httpx==0.28.1
httpx2==2.13.1
joblib==1.6.0
numpy==2.1.3
pandas==2.2.3
pandas-datareader==0.11.1
pandas-gbq==0.30.0
pandas-stubs==2.2.3.250527
pydantic==2.14.1
pydantic_core==2.50.1
safehttpx==0.1.7
scikit-learn==1.6.1
sklearn-pandas==2.2.0
uvicorn==0.54.0
Python 3.13.16


requirements.txt

In [91]:
import importlib.metadata as m
pacotes = ["fastapi", "uvicorn", "pydantic", "scikit-learn", "pandas", "numpy", "joblib", "httpx"]
with open("requirements.txt", "w") as f:
    for p in pacotes:
        f.write(f"{p}=={m.version(p)}\n")
print(open("requirements.txt").read())
!python --version

fastapi==0.141.1
uvicorn==0.54.0
pydantic==2.14.1
scikit-learn==1.6.1
pandas==2.2.3
numpy==2.1.3
joblib==1.6.0
httpx==0.28.1

Python 3.13.16


treinar_modelo.py

In [92]:
%%writefile treinar_modelo.py
import json, joblib, pandas as pd
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

df = fetch_openml("credit-g", version=1, as_frame=True).frame
df["inadimplente"] = (df["class"] == "bad").astype(int)
df = df.drop(columns=["class"])
df.to_csv("data/german_credit.csv", index=False)

num = ["duration", "credit_amount", "age"]
cat = ["purpose", "housing", "checking_status"]
X, y = df[num + cat], df["inadimplente"]

X_tmp, X_te, y_tmp, y_te = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)
X_tr, X_va, y_tr, y_va = train_test_split(X_tmp, y_tmp, test_size=0.25, random_state=42, stratify=y_tmp)

pre = ColumnTransformer([
    ("num", Pipeline([("imp", SimpleImputer(strategy="median")), ("esc", StandardScaler())]), num),
    ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")),
                      ("ohe", OneHotEncoder(handle_unknown="ignore"))]), cat),
])
modelo = Pipeline([("pre", pre), ("clf", RandomForestClassifier(
    n_estimators=200, max_depth=8, class_weight="balanced", random_state=42))])
modelo.fit(X_tr, y_tr)

def avaliar(X_, y_):
    return {"acuracia": round(accuracy_score(y_, modelo.predict(X_)), 4),
            "f1_score": round(f1_score(y_, modelo.predict(X_)), 4),
            "roc_auc": round(roc_auc_score(y_, modelo.predict_proba(X_)[:, 1]), 4)}

metricas = {"validacao": avaliar(X_va, y_va), "teste": avaliar(X_te, y_te)}
print(json.dumps(metricas, indent=2))
joblib.dump(modelo, "models/modelo_pipeline.joblib")
json.dump(metricas, open("models/metricas.json", "w"), indent=2)

Overwriting treinar_modelo.py


In [93]:
!zip -r projeto.zip app models data requirements.txt treinar_modelo.py

  adding: app/ (stored 0%)
  adding: app/__init__.py (stored 0%)
  adding: app/schemas/ (stored 0%)
  adding: app/schemas/__init__.py (stored 0%)
  adding: app/schemas/predicao.py (deflated 56%)
  adding: app/schemas/__pycache__/ (stored 0%)
  adding: app/schemas/__pycache__/__init__.cpython-313.pyc (deflated 23%)
  adding: app/schemas/__pycache__/predicao.cpython-313.pyc (deflated 33%)
  adding: app/routers/ (stored 0%)
  adding: app/routers/__init__.py (stored 0%)
  adding: app/routers/predict.py (deflated 49%)
  adding: app/routers/health.py (deflated 35%)
  adding: app/routers/__pycache__/ (stored 0%)
  adding: app/routers/__pycache__/predict.cpython-313.pyc (deflated 30%)
  adding: app/routers/__pycache__/health.cpython-313.pyc (deflated 30%)
  adding: app/routers/__pycache__/__init__.cpython-313.pyc (deflated 23%)
  adding: app/__pycache__/ (stored 0%)
  adding: app/__pycache__/main.cpython-313.pyc (deflated 36%)
  adding: app/__pycache__/__init__.cpython-313.pyc (deflated 25%)
 

In [94]:
!git clone https://github.com/Thays-Faleiro/api-risco-credito.git
%cd api-risco-credito
!git checkout feature/estrutura-api-fastapi

Cloning into 'api-risco-credito'...
remote: Enumerating objects: 18, done.
remote: Counting objects: 100% (18/18), done.
remote: Compressing objects: 100% (18/18), done.
remote: Total 18 (delta 5), reused 2 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (18/18), 11.87 KiB | 11.88 MiB/s, done.
Resolving deltas: 100% (5/5), done.
/content/api-risco-credito/api-risco-credito
branch 'feature/estrutura-api-fastapi' set up to track 'origin/feature/estrutura-api-fastapi'.
Switched to a new branch 'feature/estrutura-api-fastapi'


In [97]:

!cp -r ../app ../models ../data ../requirements.txt ../treinar_modelo.py .

# Configuracao
!git config --global user.email "thays_faleiro@estudante.sesisenai.org.br"
!git config --global user.name "Thays Faleiro"

# Executa os comandos do Git com o prefixo '!'
!git add models/ treinar_modelo.py data/
!git commit -m "implementa pipeline de treino e serializa modelo"

!git add app/schemas/
!git commit -m "implementa schemas pydantic"

!git add app/services/
!git commit -m "implementa servico de inferencia"

!git add app/routers/
!git commit -m "implementa rotas health e predict"

!git add app/main.py app/__init__.py
!git commit -m "implementa lifespan e main"

!git add requirements.txt
!git commit -m "adiciona requirements"

# Nota: O push solicitará credenciais (token do GitHub) a menos que as configurações globais ou chaves SSH estejam prontas
# Se houver erro de autenticação no push, será necessário configurar o token de acesso (PAT) do GitHub
!git push origin feature/estrutura-api-fastapi

On branch feature/estrutura-api-fastapi
Your branch is ahead of 'origin/feature/estrutura-api-fastapi' by 6 commits.
  (use "git push" to publish your local commits)

nothing to commit, working tree clean
On branch feature/estrutura-api-fastapi
Your branch is ahead of 'origin/feature/estrutura-api-fastapi' by 6 commits.
  (use "git push" to publish your local commits)

nothing to commit, working tree clean
On branch feature/estrutura-api-fastapi
Your branch is ahead of 'origin/feature/estrutura-api-fastapi' by 6 commits.
  (use "git push" to publish your local commits)

nothing to commit, working tree clean
On branch feature/estrutura-api-fastapi
Your branch is ahead of 'origin/feature/estrutura-api-fastapi' by 6 commits.
  (use "git push" to publish your local commits)

nothing to commit, working tree clean
On branch feature/estrutura-api-fastapi
Your branch is ahead of 'origin/feature/estrutura-api-fastapi' by 6 commits.
  (use "git push" to publish your local commits)

nothing to co

In [100]:
import importlib.metadata as m
pacotes = ["fastapi", "uvicorn", "pydantic", "scikit-learn", "pandas", "numpy", "joblib", "httpx"]
with open("/content/requirements.txt", "w") as f:
    for p in pacotes:
        f.write(f"{p}=={m.version(p)}\n")
print(open("/content/requirements.txt").read())
!python --version

fastapi==0.141.1
uvicorn==0.54.0
pydantic==2.14.1
scikit-learn==1.6.1
pandas==2.2.3
numpy==2.1.3
joblib==1.6.0
httpx==0.28.1

Python 3.13.16


In [101]:
!pwd
!ls /content

/content/api-risco-credito/api-risco-credito
api-risco-credito  data    projeto.zip	     sample_data
app		   models  requirements.txt  treinar_modelo.py
